# XGBoost — Otimização com Optuna & Análise de Threshold

Modelo selecionado: **XGBoost** (melhor AP e F1 no baseline)

Etapas:
1. Baseline XGBoost (referência, mesmos params do notebook anterior)
2. Optuna — busca de hiperparâmetros (objetivo: Average Precision via CV)
3. Comparação gráfica Baseline vs Otimizado
4. Otimização de Threshold — maximizar recall de fraude com controle de falsos positivos
5. Curvas Precision-Recall com threshold marcado
6. Feature Importance
7. SHAP — explicabilidade global e local
8. Resumo & Registro MLflow

**Análises complementares (adicionadas):**
- Brier Score & Log Loss (calibração de probabilidade)
- Fraudes por safra (sazonalidade e concept drift)
- PSI (Population Stability Index — estabilidade temporal do score)

---
## 1. Imports & configurações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    brier_score_loss, log_loss,
)

# ── Estilo global ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE      = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
COLOR_BASE   = '#2196F3'
COLOR_OPT    = '#FF5722'
COLOR_THRESH = '#4CAF50'
RANDOM_STATE = 42

print('Imports OK — Optuna', optuna.__version__, '| SHAP', shap.__version__)

---
## 2. Carga dos dados (Gold layer)

In [ ]:
X = pd.read_parquet('../data/gold/X_train.parquet')
y = pd.read_parquet('../data/gold/y_train.parquet').squeeze()

print(f'Shape X : {X.shape}')
print(f'Shape y : {y.shape}')
print(f'Taxa de fraude : {y.mean():.4%}')
print(f'\nFraudes    : {y.sum():,}')
print(f'Não-fraude : {(y == 0).sum():,}')
display(X.head(3))

---
## 3. Split treino / validação (estratificado)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Treino    : {X_train.shape[0]:,} amostras | fraude: {y_train.mean():.4%}')
print(f'Validação : {X_val.shape[0]:,} amostras | fraude: {y_val.mean():.4%}')

---
## 4. Funções auxiliares de métricas

In [ ]:
def ks_statistic(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))


def compute_metrics(model, Xv, yv, threshold=0.5):
    prob = model.predict_proba(Xv)[:, 1]
    pred = (prob >= threshold).astype(int)
    auc  = roc_auc_score(yv, prob)
    return {
        'Precision' : round(precision_score(yv, pred, zero_division=0), 4),
        'Recall'    : round(recall_score(yv, pred, zero_division=0), 4),
        'F1-Score'  : round(f1_score(yv, pred, zero_division=0), 4),
        'F2-Score'  : round(fbeta_score(yv, pred, beta=2, zero_division=0), 4),
        'AUC-ROC'   : round(auc, 4),
        'GINI'      : round(2 * auc - 1, 4),
        'KS'        : round(ks_statistic(yv, prob), 4),
        'AP'        : round(average_precision_score(yv, prob), 4),
        'Brier'     : round(brier_score_loss(yv, prob), 4),
        'LogLoss'   : round(log_loss(yv, prob), 4),
        '_prob'     : prob,
        '_pred'     : pred,
    }


print('Funções definidas.')

---
## 5. Baseline XGBoost (referência)

Mesmos hiperparâmetros do notebook de modelagem baseline — serve de régua para medir o ganho do Optuna.

In [ ]:
pipeline_base = Pipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        eval_metric='auc',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

t0 = time.time()
pipeline_base.fit(X_train, y_train)
print(f'Baseline treinado em {time.time() - t0:.1f}s')

metrics_base = compute_metrics(pipeline_base, X_val, y_val)
METRIC_DISPLAY = ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'AUC-ROC', 'GINI', 'KS', 'AP', 'Brier', 'LogLoss']

print('\n─── Validação — Baseline XGBoost (threshold=0.5) ───')
for k in METRIC_DISPLAY:
    print(f'  {k:<12}: {metrics_base[k]:.4f}')

---
## 6. Optuna — Otimização de Hiperparâmetros

**Objetivo:** maximizar Average Precision (AUC-PR) via validação cruzada estratificada 3-fold sobre o conjunto de treino.  
AP captura o trade-off Precision × Recall em todos os limiares — métrica ideal para classes desbalanceadas.  

> O conjunto de validação **não é usado** durante o Optuna — preservado exclusivamente para avaliação final.

In [ ]:
N_TRIALS = 60
CV_FOLDS = 3

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 800),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma'           : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    pipeline = Pipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', XGBClassifier(
            **params,
            eval_metric='auc',
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])

    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv,
        scoring='average_precision',
        n_jobs=1,
    )
    return scores.mean()


study = optuna.create_study(direction='maximize', study_name='xgboost_fraud')

t0 = time.time()
print(f'Iniciando Optuna — {N_TRIALS} trials × {CV_FOLDS}-fold CV ...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
elapsed = time.time() - t0

print(f'\nOptuna concluído em {elapsed/60:.1f} min')
print(f'Melhor AP (CV): {study.best_value:.4f}')
print(f'Melhor trial  : #{study.best_trial.number}')

In [ ]:
best_params = study.best_params
print('Hiperparâmetros otimizados:')
for k, v in best_params.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# Evolução do Optuna — melhor valor acumulado por trial
trial_values = [t.value for t in study.trials]
best_so_far  = np.maximum.accumulate(trial_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trial_values, color='#90CAF9', lw=1, alpha=0.7, label='Trial AP')
axes[0].plot(best_so_far, color=COLOR_OPT, lw=2, label='Melhor acumulado')
axes[0].axhline(metrics_base['AP'], color=COLOR_BASE, lw=1.5, ls='--', label=f'Baseline AP={metrics_base["AP"]:.4f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Average Precision (CV 3-fold)')
axes[0].set_title('Evolução do Optuna', fontweight='bold')
axes[0].legend(fontsize=9)

# Importância dos hiperparâmetros
param_imp = optuna.importance.get_param_importances(study)
keys   = list(param_imp.keys())
values = list(param_imp.values())
colors = [COLOR_OPT if v == max(values) else '#90CAF9' for v in values]
axes[1].barh(keys[::-1], values[::-1], color=colors[::-1], edgecolor='white')
axes[1].set_xlabel('Importância relativa')
axes[1].set_title('Importância dos Hiperparâmetros (Optuna)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. Modelo Otimizado — Treinamento Final

In [ ]:
pipeline_opt = Pipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', XGBClassifier(
        **best_params,
        eval_metric='auc',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

t0 = time.time()
pipeline_opt.fit(X_train, y_train)
print(f'Modelo otimizado treinado em {time.time() - t0:.1f}s')

metrics_opt = compute_metrics(pipeline_opt, X_val, y_val)

print('\n─── Validação — XGBoost Otimizado (threshold=0.5) ───')
for k in METRIC_DISPLAY:
    delta = metrics_opt[k] - metrics_base[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
    print(f'  {k:<12}: {metrics_opt[k]:.4f}  ({arrow} {delta:+.4f} vs baseline)')

---
## 8. Comparação Gráfica: Baseline vs Otimizado

In [ ]:
METRIC_PLOT = ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'AUC-ROC', 'AP', 'GINI', 'KS']

vals_base = [metrics_base[m] for m in METRIC_PLOT]
vals_opt  = [metrics_opt[m]  for m in METRIC_PLOT]

x     = np.arange(len(METRIC_PLOT))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

bars_b = ax.bar(x - width/2, vals_base, width, label='Baseline XGBoost',
                color=COLOR_BASE, alpha=0.85, edgecolor='white')
bars_o = ax.bar(x + width/2, vals_opt,  width, label='XGBoost Otimizado (Optuna)',
                color=COLOR_OPT, alpha=0.85, edgecolor='white')

for bar in list(bars_b) + list(bars_o):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(METRIC_PLOT, fontsize=10)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score')
ax.set_title('Baseline vs XGBoost Otimizado — Validação (threshold=0.5)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC — Baseline vs Otimizado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

prob_base = metrics_base['_prob']
prob_opt  = metrics_opt['_prob']

for ax, (prob, label, color) in zip(
    axes,
    [(prob_base, f'Baseline (AUC={metrics_base["AUC-ROC"]})', COLOR_BASE),
     (prob_opt,  f'Otimizado (AUC={metrics_opt["AUC-ROC"]})',  COLOR_OPT)]
):
    fpr, tpr, _ = roc_curve(y_val, prob)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=label)
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.fill_between(fpr, tpr, alpha=0.08, color=color)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('Curva ROC', fontweight='bold')
    ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1.02)

fig.suptitle('Curvas ROC — Baseline vs Otimizado', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Matrizes de Confusão — lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (label, m, color) in zip(
    axes,
    [('Baseline XGBoost',       metrics_base, COLOR_BASE),
     ('XGBoost Otimizado',      metrics_opt,  COLOR_OPT)]
):
    cm   = confusion_matrix(y_val, m['_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Legítima', 'Fraude'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    prec = m['Precision']; rec = m['Recall']; f1 = m['F1-Score']
    ax.set_title(f'{label}\nP={prec:.3f} | R={rec:.3f} | F1={f1:.3f}', fontweight='bold')

fig.suptitle('Matrizes de Confusão — threshold=0.5', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Distribuição de scores — Baseline vs Otimizado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_arr = np.array(y_val)
for ax, (prob, label) in zip(
    axes,
    [(prob_base, 'Baseline XGBoost'), (prob_opt, 'XGBoost Otimizado')]
):
    ax.hist(prob[y_arr == 0], bins=60, density=True,
            alpha=0.6, color='#2196F3', label='Legítima')
    ax.hist(prob[y_arr == 1], bins=60, density=True,
            alpha=0.6, color='#F44336', label='Fraude')
    ax.axvline(0.5, color='black', ls='--', lw=1.5, label='threshold=0.5')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Score (P(fraude))')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=9)

fig.suptitle('Distribuição de Score por Classe — Validação', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Otimização de Threshold

**Problema de negócio:**
- Bloquear clientes legítimos é muito custoso em termos de experiência e reputação
- Deixar fraudes passarem também tem custo financeiro e reputacional
- Queremos **maximizar a detecção de fraudes (recall)** mantendo uma **taxa de falsos positivos aceitável**

**Métrica guia:** F2-Score (pesa o recall 2× mais que a precision) — captura a preferência por detectar fraude sem ignorar o impacto nos clientes legítimos.

In [ ]:
thresholds  = np.linspace(0.01, 0.99, 200)
precisions  = []
recalls     = []
f1_scores   = []
f2_scores   = []
fpr_list    = []

y_arr = np.array(y_val)
n_neg = (y_arr == 0).sum()

for t in thresholds:
    pred = (prob_opt >= t).astype(int)
    precisions.append(precision_score(y_arr, pred, zero_division=0))
    recalls.append(recall_score(y_arr, pred, zero_division=0))
    f1_scores.append(f1_score(y_arr, pred, zero_division=0))
    f2_scores.append(fbeta_score(y_arr, pred, beta=2, zero_division=0))
    fp = ((pred == 1) & (y_arr == 0)).sum()
    fpr_list.append(fp / n_neg)

precisions = np.array(precisions)
recalls    = np.array(recalls)
f1_scores  = np.array(f1_scores)
f2_scores  = np.array(f2_scores)
fpr_list   = np.array(fpr_list)

# Thresholds ótimos
idx_f2   = np.argmax(f2_scores)
idx_f1   = np.argmax(f1_scores)

THRESH_OPT    = thresholds[idx_f2]   # threshold final: maximiza F2
THRESH_F1_OPT = thresholds[idx_f1]

print(f'Threshold F2-ótimo : {THRESH_OPT:.3f}  →  F2={f2_scores[idx_f2]:.4f} | Recall={recalls[idx_f2]:.4f} | Precision={precisions[idx_f2]:.4f} | FPR={fpr_list[idx_f2]:.4f}')
print(f'Threshold F1-ótimo : {THRESH_F1_OPT:.3f}  →  F1={f1_scores[idx_f1]:.4f} | Recall={recalls[idx_f1]:.4f} | Precision={precisions[idx_f1]:.4f} | FPR={fpr_list[idx_f1]:.4f}')
print(f'Threshold padrão   : 0.500  →  F2={fbeta_score(y_arr,(prob_opt>=0.5).astype(int),beta=2,zero_division=0):.4f} | Recall={recall_score(y_arr,(prob_opt>=0.5).astype(int),zero_division=0):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Precision & Recall vs Threshold ─────────────────────────────────────────
ax = axes[0]
ax.plot(thresholds, precisions, color='#2196F3', lw=2, label='Precision')
ax.plot(thresholds, recalls,    color='#F44336', lw=2, label='Recall')
ax.axvline(THRESH_OPT,    color=COLOR_THRESH, lw=1.8, ls='--', label=f'F2-ótimo ({THRESH_OPT:.3f})')
ax.axvline(THRESH_F1_OPT, color='#9C27B0',    lw=1.8, ls=':',  label=f'F1-ótimo ({THRESH_F1_OPT:.3f})')
ax.axvline(0.5,           color='gray',        lw=1,   ls='--', label='Padrão (0.5)')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision & Recall vs Threshold', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

# ── F1 & F2 vs Threshold ─────────────────────────────────────────────────────
ax = axes[1]
ax.plot(thresholds, f1_scores, color='#9C27B0', lw=2, label='F1-Score')
ax.plot(thresholds, f2_scores, color=COLOR_OPT, lw=2, label='F2-Score (β=2)')
ax.axvline(THRESH_OPT,    color=COLOR_THRESH, lw=1.8, ls='--', label=f'F2-ótimo ({THRESH_OPT:.3f})')
ax.axvline(THRESH_F1_OPT, color='#9C27B0',    lw=1.8, ls=':',  label=f'F1-ótimo ({THRESH_F1_OPT:.3f})')
ax.axvline(0.5,           color='gray',        lw=1,   ls='--', label='Padrão (0.5)')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('F1 & F2 vs Threshold', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

# ── FPR vs Threshold ─────────────────────────────────────────────────────────
ax = axes[2]
ax.plot(thresholds, fpr_list, color='#FF9800', lw=2, label='FPR (clientes bloqueados)')
ax.fill_between(thresholds, fpr_list, alpha=0.12, color='#FF9800')
ax.axvline(THRESH_OPT,    color=COLOR_THRESH, lw=1.8, ls='--',
           label=f'F2-ótimo — FPR={fpr_list[idx_f2]:.4f}')
ax.axvline(0.5,           color='gray',        lw=1,   ls='--',
           label=f'Padrão   — FPR={fpr_list[np.argmin(np.abs(thresholds-0.5))]:.4f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Taxa de Falsos Positivos')
ax.set_title('FPR (clientes legítimos bloqueados)', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0, 1)

fig.suptitle('Análise de Threshold — XGBoost Otimizado', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Métricas com o threshold otimizado por F2
metrics_thresh = compute_metrics(pipeline_opt, X_val, y_val, threshold=THRESH_OPT)

print(f'\n─── Comparativo de Thresholds — XGBoost Otimizado ───')
print(f'{"Métrica":<12} {"Padrão (0.5)":>14} {f"F2-ótimo ({THRESH_OPT:.3f})":>18} {"Δ":>10}')
print('─' * 58)
for k in ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'AUC-ROC', 'AP']:
    v_05 = metrics_opt[k]
    v_t  = metrics_thresh[k]
    delta = v_t - v_05
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
    print(f'{k:<12} {v_05:>14.4f} {v_t:>18.4f} {arrow} {delta:>+8.4f}')

---
## 10. Curva Precision-Recall com Threshold Marcado

In [ ]:
prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, prob_opt)
ap_opt = average_precision_score(y_val, prob_opt)
ap_base = average_precision_score(y_val, prob_base)

# Ponto no threshold F2-ótimo
prec_t = metrics_thresh['Precision']
rec_t  = metrics_thresh['Recall']
prec_05 = metrics_opt['Precision']
rec_05  = metrics_opt['Recall']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Curva PR — Baseline vs Otimizado ────────────────────────────────────────
ax = axes[0]
prec_b, rec_b, _ = precision_recall_curve(y_val, prob_base)
ax.plot(rec_b,      prec_b,      color=COLOR_BASE, lw=2,   label=f'Baseline (AP={ap_base:.4f})')
ax.plot(rec_curve,  prec_curve,  color=COLOR_OPT,  lw=2.5, label=f'Otimizado (AP={ap_opt:.4f})')
ax.fill_between(rec_curve, prec_curve, alpha=0.08, color=COLOR_OPT)
ax.axhline(y_val.mean(), color='gray', ls=':', lw=1, label=f'Baseline aleatório ({y_val.mean():.3f})')
ax.scatter([rec_05], [prec_05], color='gray',        s=100, zorder=5, label=f'threshold=0.5')
ax.scatter([rec_t],  [prec_t],  color=COLOR_THRESH,  s=120, zorder=5, label=f'threshold F2={THRESH_OPT:.3f}', marker='*')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall', fontweight='bold')
ax.legend(fontsize=8.5); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# ── Recall vs Threshold (Recall Curve) ─────────────────────────────────────
ax = axes[1]
ax.plot(thresholds, recalls,    color=COLOR_OPT,    lw=2.5, label='Recall (fraude)')
ax.plot(thresholds, precisions, color='#2196F3',    lw=2,   alpha=0.7, label='Precision')
ax.plot(thresholds, fpr_list,   color='#FF9800',    lw=2,   alpha=0.7, label='FPR (legítimos bloqueados)')

ax.axvline(THRESH_OPT, color=COLOR_THRESH, lw=2, ls='--',
           label=f'Threshold ótimo ({THRESH_OPT:.3f})\nRecall={recalls[idx_f2]:.3f} | FPR={fpr_list[idx_f2]:.4f}')
ax.axvline(0.5, color='gray', lw=1.2, ls=':', label='Padrão (0.5)')

ax.fill_betweenx([0, 1], 0, THRESH_OPT, alpha=0.04, color=COLOR_THRESH)
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Recall Curve — Detecção de Fraude vs Clientes Bloqueados', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

fig.suptitle('Análise Precision-Recall — XGBoost Otimizado', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Matriz de confusão com threshold otimizado
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('Baseline\nthreshold=0.5',       metrics_base['_pred'],   COLOR_BASE),
    ('Otimizado\nthreshold=0.5',      metrics_opt['_pred'],    COLOR_OPT),
    (f'Otimizado\nthreshold={THRESH_OPT:.3f} (F2)', metrics_thresh['_pred'], COLOR_THRESH),
]

for ax, (title, pred, color) in zip(axes, configs):
    cm   = confusion_matrix(y_val, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Legítima', 'Fraude'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(
        f'{title}\nTP={tp} | FN={fn} | FP={fp} | TN={tn}',
        fontweight='bold', fontsize=9
    )

fig.suptitle('Matrizes de Confusão — Comparativo de Thresholds', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 11. Feature Importance — XGBoost Otimizado

In [ ]:
xgb_model     = pipeline_opt.named_steps['model']
feature_names = X_train.columns.tolist()

# Três tipos de importância do XGBoost
importance_types = {
    'weight'   : 'Frequência (weight) — nº de splits',
    'gain'     : 'Ganho médio (gain) — redução de impureza',
    'cover'    : 'Cobertura (cover) — nº de amostras afetadas',
}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, (itype, title) in zip(axes, importance_types.items()):
    scores = xgb_model.get_booster().get_score(importance_type=itype)
    df_imp = (
        pd.Series(scores, name='importance')
        .reindex(feature_names, fill_value=0)
        .sort_values(ascending=True)
    )
    top = df_imp.tail(20)
    colors = [COLOR_OPT if v == top.max() else '#90CAF9' for v in top.values]
    top.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Importância')
    ax.set_ylabel('')

fig.suptitle('Feature Importance — XGBoost Otimizado (Top 20)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 12. SHAP — Explicabilidade Global e Local

SHAP (SHapley Additive exPlanations) decompõe a predição de cada transação em contribuições individuais de cada feature.  
Responde: *"por que o modelo classificou esta transação como fraude?"*

In [ ]:
# Amostra de validação para SHAP (máx 5000 para agilidade)
N_SHAP   = min(5000, len(X_val))
idx_shap = np.random.RandomState(RANDOM_STATE).choice(len(X_val), N_SHAP, replace=False)
X_shap   = X_val.iloc[idx_shap].reset_index(drop=True)
y_shap   = y_val.iloc[idx_shap].reset_index(drop=True)

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_shap)

print(f'SHAP calculado para {N_SHAP} amostras')
print(f'Fraudes na amostra SHAP: {y_shap.sum()} ({y_shap.mean():.2%})')

In [ ]:
# SHAP Summary — Beeswarm (impacto por feature por amostra)
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('SHAP Beeswarm — Impacto de cada feature na predição de fraude',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar — importância global média |SHAP|
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title('SHAP Bar — Importância Global (média |SHAP|)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Waterfall — explicação local de um caso de fraude
fraud_indices = np.where(np.array(y_shap) == 1)[0]
if len(fraud_indices) > 0:
    idx_fraud = fraud_indices[0]
    shap.plots.waterfall(shap_values[idx_fraud], max_display=15, show=False)
    prob_caso = explainer.model.predict(
        xgb_model.get_booster(),
        enable_categorical=False
    ) if False else pipeline_opt.predict_proba(X_shap.iloc[[idx_fraud]])[:, 1][0]
    plt.title(f'SHAP Waterfall — Caso de Fraude (score={prob_caso:.3f})',
              fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()
else:
    print('Nenhuma fraude na amostra SHAP — aumente N_SHAP.')

In [ ]:
# Top features por SHAP — heatmap de contribuições
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
df_shap_imp = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)
top10_features = df_shap_imp.head(10).index.tolist()

shap_matrix = pd.DataFrame(
    shap_values.values[:, [feature_names.index(f) for f in top10_features]],
    columns=top10_features
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    shap_matrix.sample(min(200, len(shap_matrix)), random_state=RANDOM_STATE).T,
    cmap='coolwarm', center=0, ax=ax,
    xticklabels=False, yticklabels=True,
    cbar_kws={'label': 'SHAP value'}
)
ax.set_title('SHAP Heatmap — Top 10 Features (amostra de 200 transações)',
             fontweight='bold')
ax.set_xlabel('Transações'); ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## 14. Estabilidade Temporal, Fraudes por Safra & Calibração

Três perspectivas complementares que avaliam **robustez ao longo do tempo** e **qualidade probabilística** do modelo:

- **Fraudes por safra** — volume absoluto e taxa de fraude por período, revelando sazonalidades e concept drift no alvo  
- **PSI (Population Stability Index)** — mede se a distribuição de scores muda entre safras; limiar clássico: PSI < 0.10 estável, 0.10–0.25 atenção, > 0.25 instável  
- **Brier Score & Log Loss** — medem qualidade da probabilidade calibrada (quanto menor, melhor); avaliados globalmente e por safra para detectar degradação temporal

In [ ]:
# ── Safra: TX_DATETIME de train_silver (coluna temporal original do dataset) ──
# X_train.parquet contém apenas as 29 features selecionadas pelo feature_selector.
# TX_DATETIME não entra no modelo, mas está sempre disponível no silver.

_silver_temporal = pd.read_parquet(
    '../data/silver/train_silver.parquet',
    columns=['TX_DATETIME'],
)

# Série de safra derivada de TX_DATETIME (formato YYYY-MM)
_safra_series = _silver_temporal['TX_DATETIME'].dt.to_period('M').astype(str)
_safra_series.index = X.index   # garante alinhamento com os índices de X

SAFRA_COL = 'safra'

# X_val foi criado antes deste ponto (train_test_split na seção 3), então
# criamos uma cópia específica para a análise de safra, sem alterar X_val.
X_val_safra           = X_val.copy()
X_val_safra[SAFRA_COL] = _safra_series.loc[X_val.index].values

print(f'Safra: TX_DATETIME → período mensal (YYYY-MM)')
print(f'  Safras únicas : {X_val_safra[SAFRA_COL].nunique()}')
print(f'  Valores       : {sorted(X_val_safra[SAFRA_COL].unique())}')
print()
print(X_val_safra[SAFRA_COL].value_counts().sort_index().rename('n_transações').to_string())

In [ ]:
# ── Fraudes por Safra ─────────────────────────────────────────────────────────
assert SAFRA_COL is not None, 'Defina SAFRA_COL manualmente antes de continuar.'

# Reconstrói dataset de validação com a coluna de safra
df_val = X_val_safra[[SAFRA_COL]].copy()
df_val.index = range(len(df_val))  # reset para alinhamento com arrays
df_val['target']     = y_val.values
df_val['score']      = prob_opt            # scores do modelo otimizado (threshold F2)

safra_stats = (
    df_val.groupby(SAFRA_COL)
    .agg(
        total         = ('target', 'count'),
        fraudes       = ('target', 'sum'),
        taxa_fraude   = ('target', 'mean'),
        score_medio   = ('score',  'mean'),
    )
    .reset_index()
    .sort_values(SAFRA_COL)
)
safra_stats['taxa_fraude_pct'] = safra_stats['taxa_fraude'] * 100

print(safra_stats.to_string(index=False))


In [ ]:
# ── Gráfico: Volume e Taxa de Fraude por Safra ────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={'height_ratios': [1.6, 1]})

safras = safra_stats[SAFRA_COL].astype(str)
x      = np.arange(len(safras))

# — Painel superior: volumes
ax1 = axes[0]
bars_leg  = ax1.bar(x, safra_stats['total'] - safra_stats['fraudes'],
                    color='#90CAF9', alpha=0.85, label='Legítimas', edgecolor='white')
bars_frau = ax1.bar(x, safra_stats['fraudes'],
                    bottom=safra_stats['total'] - safra_stats['fraudes'],
                    color='#EF9A9A', alpha=0.9, label='Fraudes', edgecolor='white')

for bar, fq in zip(bars_frau, safra_stats['fraudes']):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_y() + bar.get_height() + safra_stats['total'].max() * 0.01,
             str(int(fq)), ha='center', va='bottom', fontsize=8, color='#C62828')

ax1.set_ylabel('Transações')
ax1.set_title('Volume de Transações por Safra (Validação)', fontweight='bold')
ax1.legend(fontsize=9)

# — Painel inferior: taxa de fraude
ax2 = axes[1]
ax2.plot(x, safra_stats['taxa_fraude_pct'], color='#F44336',
         marker='o', lw=2.5, ms=7, label='Taxa de fraude (%)')
ax2.fill_between(x, safra_stats['taxa_fraude_pct'], alpha=0.10, color='#F44336')
ax2.axhline(safra_stats['taxa_fraude_pct'].mean(), color='gray',
            ls='--', lw=1.2, label=f'Média {safra_stats["taxa_fraude_pct"].mean():.2f}%')

ax2.set_ylabel('Taxa de Fraude (%)')
ax2.set_xlabel('Safra')
ax2.set_xticks(x)
ax2.set_xticklabels(safras, rotation=30, ha='right', fontsize=9)
ax2.legend(fontsize=9)

fig.suptitle('Distribuição Temporal de Fraudes — Conjunto de Validação',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── PSI — Population Stability Index por Safra ───────────────────────────────
def psi_score(ref: np.ndarray, cur: np.ndarray, n_bins: int = 10) -> float:
    """Calcula PSI entre distribuição de referência e atual (scores 0-1)."""
    bins    = np.linspace(0, 1, n_bins + 1)
    ref_pct = np.histogram(ref, bins=bins)[0] / len(ref)
    cur_pct = np.histogram(cur, bins=bins)[0] / len(cur)
    # Suavização para evitar log(0)
    ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
    cur_pct = np.where(cur_pct == 0, 1e-6, cur_pct)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


# Safra mais antiga como referência
safras_ord = safra_stats[SAFRA_COL].tolist()
ref_safra  = safras_ord[0]
ref_scores = df_val.loc[df_val[SAFRA_COL] == ref_safra, 'score'].values

psi_results = []
for safra in safras_ord:
    cur_scores = df_val.loc[df_val[SAFRA_COL] == safra, 'score'].values
    psi_val    = psi_score(ref_scores, cur_scores)
    status     = ('✅ Estável'   if psi_val < 0.10
                  else '⚠️ Atenção'  if psi_val < 0.25
                  else '🔴 Instável')
    psi_results.append({'Safra': safra, 'PSI': round(psi_val, 4), 'Status': status})

df_psi = pd.DataFrame(psi_results)
print(f'Safra de referência: {ref_safra}')
print()
print(df_psi.to_string(index=False))

# ── Plot PSI ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

colors_psi = ['#4CAF50' if v < 0.10 else '#FF9800' if v < 0.25 else '#F44336'
              for v in df_psi['PSI']]

bars = ax.bar(np.arange(len(df_psi)), df_psi['PSI'], color=colors_psi,
              edgecolor='white', alpha=0.88)

ax.axhline(0.10, color='#FF9800', ls='--', lw=1.5, label='Limite atenção (0.10)')
ax.axhline(0.25, color='#F44336', ls='--', lw=1.5, label='Limite instável (0.25)')

for bar, val in zip(bars, df_psi['PSI']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(np.arange(len(df_psi)))
ax.set_xticklabels(df_psi['Safra'].astype(str), rotation=30, ha='right', fontsize=9)
ax.set_ylabel('PSI')
ax.set_title(f'Population Stability Index por Safra (ref: {ref_safra})',
             fontweight='bold')
ax.legend(fontsize=9)

# Legenda de cores
from matplotlib.patches import Patch
handles = [Patch(facecolor='#4CAF50', label='Estável  (< 0.10)'),
           Patch(facecolor='#FF9800', label='Atenção  (0.10–0.25)'),
           Patch(facecolor='#F44336', label='Instável (≥ 0.25)')]
ax.legend(handles=handles, fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# ── Brier Score & Log Loss por Safra ─────────────────────────────────────────
from sklearn.metrics import brier_score_loss, log_loss

brier_global = brier_score_loss(df_val['target'], df_val['score'])
ll_global    = log_loss(df_val['target'], df_val['score'])

print(f'Brier Score global  : {brier_global:.4f}   (0 = perfeito, 0.25 = aleatório)')
print(f'Log Loss global     : {ll_global:.4f}   (0 = perfeito)')
print()

calib_per_safra = []
for safra in safras_ord:
    mask  = df_val[SAFRA_COL] == safra
    yt    = df_val.loc[mask, 'target'].values
    yp    = df_val.loc[mask, 'score'].values
    if len(np.unique(yt)) < 2:
        continue
    calib_per_safra.append({
        'Safra'  : safra,
        'N'      : int(mask.sum()),
        'Fraudes': int(yt.sum()),
        'Brier'  : round(brier_score_loss(yt, yp), 4),
        'LogLoss': round(log_loss(yt, yp), 4),
    })

df_calib = pd.DataFrame(calib_per_safra)
print(df_calib.to_string(index=False))

# ── Plot Brier & LogLoss por Safra ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (col, color, label) in zip(
    axes,
    [('Brier',   '#FF5722', 'Brier Score'),
     ('LogLoss', '#9C27B0', 'Log Loss')]
):
    xs = np.arange(len(df_calib))
    ax.bar(xs, df_calib[col], color=color, alpha=0.75, edgecolor='white')
    ax.axhline(df_calib[col].mean(), color='gray', ls='--', lw=1.4,
               label=f'Média {df_calib[col].mean():.4f}')
    ax.axhline(brier_global if col == 'Brier' else ll_global,
               color='black', ls=':', lw=1.5, label='Global')

    for i, (val, n) in enumerate(zip(df_calib[col], df_calib['N'])):
        ax.text(i, val + ax.get_ylim()[1] * 0.01 if ax.get_ylim()[1] > 0 else 0,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(xs)
    ax.set_xticklabels(df_calib['Safra'].astype(str), rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f'{label} por Safra', fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Qualidade de Calibração por Safra — XGBoost Otimizado',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 15. Resumo Final & Registro MLflow

In [ ]:
METRIC_DISPLAY = ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'AUC-ROC', 'AP', 'GINI', 'KS', 'Brier', 'LogLoss']

df_summary = pd.DataFrame({
    'Baseline (thr=0.5)':            {k: metrics_base[k]   for k in METRIC_DISPLAY},
    'Otimizado (thr=0.5)':           {k: metrics_opt[k]    for k in METRIC_DISPLAY},
    f'Otimizado (thr={THRESH_OPT:.3f})': {k: metrics_thresh[k] for k in METRIC_DISPLAY},
}).T

display(
    df_summary.style
    .format('{:.4f}')
    .background_gradient(cmap='RdYlGn', axis=0)
    .set_caption('Resumo — Baseline vs Otimizado vs Threshold ajustado')
    .set_table_styles([{'selector': 'caption',
                        'props': 'font-size:14px; font-weight:bold; text-align:left;'}])
)

print(f'\nThreshold selecionado: {THRESH_OPT:.3f} (F2-Score ótimo)')
print(f'  Recall de fraude  : {metrics_thresh["Recall"]:.4f}  → detecta {metrics_thresh["Recall"]*100:.1f}% das fraudes')
print(f'  Precision         : {metrics_thresh["Precision"]:.4f}')
print(f'  FPR               : {fpr_list[idx_f2]:.4f}  → bloqueia {fpr_list[idx_f2]*100:.2f}% dos clientes legítimos')

In [ ]:
import sys, os

# Garante que o cwd e a raiz do projeto (necessario ao rodar de notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from src.context.pipeline_context import PipelineContext
from src.agents.ExperimentLoggerAgent import ExperimentLoggerAgent
from src.hooks.logging_hooks import pre_logging_hook, post_logging_hook

# feature_names ja esta disponivel no escopo (definido na secao 11)
ctx                 = PipelineContext()
ctx.model           = pipeline_opt
ctx.model_name      = f"xgboost_optuna_thr{THRESH_OPT:.3f}".replace(".", "_")
ctx.threshold       = THRESH_OPT
ctx.feature_columns = feature_names
ctx.metrics         = {
    "auc_roc"           : metrics_thresh["AUC-ROC"],
    "average_precision" : metrics_thresh["AP"],
    "f1_fraud"          : metrics_thresh["F1-Score"],
    "f2_fraud"          : metrics_thresh["F2-Score"],
    "precision_fraud"   : metrics_thresh["Precision"],
    "recall_fraud"      : metrics_thresh["Recall"],
    "gini"              : metrics_thresh["GINI"],
    "ks"                : metrics_thresh["KS"],
}

pre_logging_hook(ctx)
if not ctx.has_errors():
    t0  = time.time()
    ctx = ExperimentLoggerAgent(experiment_name="fraud-detection-optuna").run(ctx)
    post_logging_hook(ctx, t0)
    print(f"
run_id: {ctx.run_id}")
else:
    print("Bloqueado pelo pre_logging_hook:", ctx.errors)
